# OBD-2 Engine Health — Exploratory Data Analysis
Run this notebook **before** `preprocess.py` to understand your data and tune label thresholds.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Data
Put your downloaded Kaggle CSV in `data/raw/`. Update the filename below.

In [ ]:
df = pd.read_csv('../data/raw/engine_data.csv')   # <-- change filename
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Column info
df.info()

In [ ]:
# Basic statistics
df.describe().round(2)

## 2. Missing Values

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing) == 0:
    print('No missing values.')
else:
    print(missing)
    fig, ax = plt.subplots(figsize=(8, 4))
    missing.plot(kind='bar', ax=ax, color='salmon')
    ax.set_title('Missing Values per Column')
    plt.tight_layout()

## 3. Class Distribution (if label column exists)

In [ ]:
# Change 'Engine Condition' to your actual label column name
LABEL_COL = 'Engine Condition'

if LABEL_COL in df.columns:
    counts = df[LABEL_COL].value_counts()
    print(counts)
    fig, ax = plt.subplots(figsize=(6, 4))
    counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
    ax.set_title('Class Distribution')
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    plt.xticks(rotation=0)
    plt.tight_layout()
else:
    print(f'Column "{LABEL_COL}" not found. Auto-labeling will be used.')

## 4. Feature Distributions

In [ ]:
# Update this list to match your actual column names
FEATURE_COLS = ['Engine rpm', 'Lub oil pressure', 'Fuel pressure', 'Coolant temp', 'lub oil temp']
available = [c for c in FEATURE_COLS if c in df.columns]

fig, axes = plt.subplots(1, len(available), figsize=(4 * len(available), 4))
if len(available) == 1:
    axes = [axes]
for ax, col in zip(axes, available):
    ax.hist(df[col].dropna(), bins=50, color='steelblue', edgecolor='white')
    ax.set_title(col, fontsize=9)
    ax.set_xlabel('Value')
plt.suptitle('Feature Distributions', y=1.02)
plt.tight_layout()

## 5. Box Plots by Class

In [ ]:
if LABEL_COL in df.columns and len(available) > 0:
    fig, axes = plt.subplots(1, len(available), figsize=(4 * len(available), 5))
    if len(available) == 1:
        axes = [axes]
    for ax, col in zip(axes, available):
        df.boxplot(column=col, by=LABEL_COL, ax=ax)
        ax.set_title(col, fontsize=9)
        ax.set_xlabel('Class')
    plt.suptitle('Feature Values by Engine Condition')
    plt.tight_layout()

## 6. Correlation Heatmap

In [ ]:
numeric_df = df[available].copy()
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, vmin=-1, vmax=1, ax=ax,
            square=True, linewidths=0.5)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()

## 7. Outlier Check (IQR)

In [ ]:
for col in available:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_outliers = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    pct = 100 * n_outliers / len(df)
    print(f'{col:30s}  outliers: {n_outliers:5d}  ({pct:.1f}%)')

## 8. Suggest Label Thresholds
Use percentiles to set sensible thresholds for `config.yaml`.

In [ ]:
for col in available:
    p90 = df[col].quantile(0.90)
    p95 = df[col].quantile(0.95)
    p99 = df[col].quantile(0.99)
    print(f'{col:30s}  p90={p90:.2f}  p95={p95:.2f}  p99={p99:.2f}')

print('\n→ Use p90 as Warning threshold, p95 as Faulty, p99 as Critical in config.yaml')

## 9. Time Series Preview (first 500 rows)

In [ ]:
sample = df[available].head(500)
fig, axes = plt.subplots(len(available), 1, figsize=(14, 2.5 * len(available)), sharex=True)
if len(available) == 1:
    axes = [axes]
for ax, col in zip(axes, available):
    ax.plot(sample[col].values, lw=0.8, color='steelblue')
    ax.set_ylabel(col, fontsize=8)
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('Sample index')
plt.suptitle('Time Series Preview (first 500 rows)', y=1.01)
plt.tight_layout()